# 🏦 LangGraph HITL — Loan Approval Workflow
**Extension 2** | Pattern: `planner_multi_agent` + Human-in-the-Loop

---

## Architecture

```
planner ──► executor ──► verifier
                │            │
           (loop back)   approved
           max 3 iters       │
                         route_by_risk()
                         /           \
                  auto_process    human_review
                  (low risk)      ⏸ interrupt()
                                  Command(resume=...)
                                       │
                                      END
```

| Node | Role |
|------|------|
| `planner` | LLM breaks goal into ≤5 tasks |
| `executor` | LLM executes tasks + routes loans by risk/amount |
| `verifier` | LLM-as-judge; loops back if quality score low |
| `auto_process` | Rule-based approval for low-risk loans |
| `human_review` | `interrupt()` pause for high-risk/high-value loans |

**Routing:** `loan_amount ≥ INR 10L` or `risk_score ≥ 0.70` → `human_review`

> 🔑 **Groq API key required** — https://console.groq.com/keys


## Cell 1 — Install

In [ ]:
%%capture
!pip install langgraph langchain-core langchain-groq langchain-community python-dotenv
!pip install -U ddgs
print("✅ Done")


## Cell 2 — Groq API Key
Powers the **planner**, **executor**, and **verifier** nodes (`llama-3.1-8b-instant`).


In [ ]:
import os

GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"  # ← paste your key here
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("✅ API key set")


## Cell 3 — Imports

In [ ]:
import os, json
from enum import Enum
from dataclasses import dataclass
from typing import TypedDict, List, Optional

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

print("✅ Imports done")


## Cell 4 — Domain Types & Shared State
`LoanState` extends `AgentState` from the reference notebook with loan-specific fields.
Each node receives the full state and returns only the fields it changes.


In [ ]:
class ReviewDecision(Enum):
    APPROVE  = "approve"
    REJECT   = "reject"
    ESCALATE = "escalate"


@dataclass
class LoanRequest:
    application_id:    str
    loan_amount:       float
    applicant_summary: str
    ai_recommendation: str
    risk_score:        float
    review_required:   bool                     = False
    human_decision:    Optional[ReviewDecision] = None
    reviewer_id:       Optional[str]            = None
    review_notes:      Optional[str]            = None


class LoanState(TypedDict):
    # reference fields (same as AgentState in planner_multi_agent)
    goal:            str
    tasks:           List[str]
    results:         List[str]
    critique:        str
    approved:        bool
    iterations:      int
    # loan-specific fields
    loans:           List[dict]
    review_queue:    List[dict]
    auto_queue:      List[dict]
    final_decisions: List[dict]


HIGH_VALUE_THRESHOLD = 1_000_000  # INR 10 lakhs
HIGH_RISK_SCORE      = 0.7

print("✅ Types and state defined")


## Cell 5 — LLM & Tools

In [ ]:
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.1-8b-instant",
    groq_api_key=GROQ_API_KEY
)
search = DuckDuckGoSearchRun()

print("✅ LLM: llama-3.1-8b-instant (Groq)")
print("✅ Search: DuckDuckGo")


## Cell 6 — Node 1: `planner`
LLM breaks the goal into ≤5 tasks → `state["tasks"]`.


In [ ]:
def planner(state: LoanState) -> dict:
    system = (
        "You are a planning agent for a loan approval workflow. "
        "Break the goal into at most 5 concrete, actionable tasks. "
        "Respond ONLY with a valid JSON array of strings. No preamble, no markdown."
    )
    response = llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"Goal: {state['goal']}")
    ]).content.strip()

    try:
        tasks = json.loads(response.replace("```json", "").replace("```", "").strip())
    except json.JSONDecodeError:
        tasks = [response]

    print(f"[Planner] {len(tasks)} tasks:")
    for i, t in enumerate(tasks, 1):
        print(f"  {i}. {t}")

    return {**state, "tasks": tasks}


print("✅ planner defined")


## Cell 7 — Node 2: `executor`
Executes each task via LLM + optional DuckDuckGo search, then routes loans by risk thresholds.


In [ ]:
def executor(state: LoanState) -> dict:
    results      = []
    review_queue = []
    auto_queue   = []
    critique_ctx = f"\nPrevious critique: {state['critique']}" if state["critique"] else ""

    for task in state["tasks"]:
        search_ctx = ""
        try:
            search_ctx = f"\nWeb search:\n{search.run(task[:100])[:800]}"
        except Exception:
            pass

        result = llm.invoke([
            SystemMessage(content=f"You are an execution agent for a loan approval process. Complete the task thoroughly.{critique_ctx}"),
            HumanMessage(content=f"Task: {task}{search_ctx}")
        ]).content
        results.append(result)
        print(f"[Executor] {task[:60]}\n  → {result[:150]}\n")

    # HITL routing
    print("─" * 60)
    for loan in state["loans"]:
        needs_review = (
            loan["loan_amount"] >= HIGH_VALUE_THRESHOLD
            or loan["risk_score"] >= HIGH_RISK_SCORE
        )
        loan["review_required"] = needs_review
        if needs_review:
            review_queue.append(loan)
            print(f"🔄 HUMAN_REVIEW  [{loan['application_id']}] INR {loan['loan_amount']:,.0f} | Risk {loan['risk_score']:.2f}")
        else:
            auto_queue.append(loan)
            print(f"⚡ AUTO_PROCESS  [{loan['application_id']}] INR {loan['loan_amount']:,.0f} | Risk {loan['risk_score']:.2f}")

    return {**state, "results": results, "review_queue": review_queue,
            "auto_queue": auto_queue, "iterations": state["iterations"] + 1}


print("✅ executor defined")


## Cell 8 — Node 3: `verifier`
LLM-as-judge scores results (0–1.0). Loops back to executor if score low. Force-approves after 3 iterations.


In [ ]:
def verifier(state: LoanState) -> dict:
    if state["iterations"] >= 3:
        print("[Verifier] Max iterations — force approving.")
        return {**state, "approved": True}

    combined = "\n\n".join(
        f"Task {i+1}: {t}\nResult: {r}"
        for i, (t, r) in enumerate(zip(state["tasks"], state["results"]))
    )
    system = (
        "You are a quality verifier. Score results (0.0-1.0) on:\n"
        "- Completeness (0-0.4), Accuracy (0-0.3), Clarity (0-0.3)\n"
        'Respond ONLY as JSON: {"score":0.85,"approved":true,"critique":"..."}'
    )
    raw = llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"Goal: {state['goal']}\n\nResults:\n{combined}")
    ]).content.strip()

    try:
        verdict  = json.loads(raw.replace("```json","").replace("```","").strip())
        approved = verdict.get("approved", False)
        critique = verdict.get("critique", "")
        score    = verdict.get("score", 0)
    except Exception:
        approved, critique, score = False, raw, 0

    print(f"[Verifier] Score: {score:.2f} | Approved: {approved}")
    if not approved:
        print(f"  Critique: {critique}")

    return {**state, "approved": approved, "critique": critique}


def route_after_verify(state: LoanState) -> str:
    return "end_routing" if state["approved"] else "executor"


print("✅ verifier defined")


## Cell 9 — Node 4: `auto_process`
Auto-approves all low-risk/low-value loans. Then checks if any loans still need human review.


In [ ]:
def auto_process(state: LoanState) -> dict:
    final_decisions = list(state.get("final_decisions", []))
    print("⚡ Auto-processing...")
    for loan in state["auto_queue"]:
        final_decisions.append({
            **loan,
            "human_decision": ReviewDecision.APPROVE.value,
            "reviewer_id":    "AUTO_SYSTEM",
            "review_notes":   "Auto-approved: below thresholds",
        })
        print(f"  ✅ [{loan['application_id']}] APPROVED")
    return {**state, "final_decisions": final_decisions}


def route_by_risk(state: LoanState) -> str:
    return "human_review" if state.get("review_queue") else "done"


print("✅ auto_process defined")


## Cell 10 — Node 5: `human_review`
`interrupt()` pauses the graph and surfaces the review queue to the caller.  
Graph resumes when `Command(resume={"decisions": [...]})` is passed back.


In [ ]:
def human_review(state: LoanState) -> dict:
    print("\n" + "═" * 60)
    print(f"⏸  PAUSED — {len(state['review_queue'])} loan(s) pending:")
    for loan in state["review_queue"]:
        print(f"   [{loan['application_id']}] INR {loan['loan_amount']:,.0f} | Risk {loan['risk_score']:.2f} | {loan['ai_recommendation']}")
    print("═" * 60)

    human_input     = interrupt({"message": "Awaiting underwriter decisions.", "review_queue": state["review_queue"]})
    decisions       = human_input.get("decisions", [])
    final_decisions = list(state.get("final_decisions", []))

    print("\n👤 Decisions received:")
    for loan in state["review_queue"]:
        match = next((d for d in decisions if d["application_id"] == loan["application_id"]), None)
        if match:
            final_decisions.append({**loan, "human_decision": match["decision"],
                                     "reviewer_id": match.get("reviewer_id","UNKNOWN"),
                                     "review_notes": match.get("notes","")})
            print(f"   [{loan['application_id']}] → {match['decision'].upper()} by {match.get('reviewer_id','UNKNOWN')}")
        else:
            final_decisions.append({**loan, "human_decision": ReviewDecision.ESCALATE.value,
                                     "reviewer_id": "SYSTEM", "review_notes": "No decision — escalated"})
            print(f"   [{loan['application_id']}] → ESCALATED")

    return {**state, "final_decisions": final_decisions}


print("✅ human_review defined")


## Cell 11 — Build & Compile Graph

In [ ]:
graph = StateGraph(LoanState)

graph.add_node("planner",      planner)
graph.add_node("executor",     executor)
graph.add_node("verifier",     verifier)
graph.add_node("auto_process", auto_process)
graph.add_node("human_review", human_review)

graph.set_entry_point("planner")
graph.add_edge("planner",  "executor")
graph.add_edge("executor", "verifier")
graph.add_conditional_edges("verifier",     route_after_verify, {"executor": "executor", "end_routing": "auto_process"})
graph.add_conditional_edges("auto_process", route_by_risk,      {"human_review": "human_review", "done": END})
graph.add_edge("human_review", END)

app = graph.compile(checkpointer=MemorySaver())
print("✅ Graph compiled")
print("   planner → executor → verifier → auto_process → [human_review] → END")


## Cell 12 — Visualise Graph *(optional)*

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())


## Cell 13 — Test Data

| ID | Amount (INR) | Risk | Expected Route |
|----|-------------|------|----------------|
| APP100301 | 5,00,000 | 0.25 | auto_process |
| APP100302 | 15,00,000 | 0.62 | human_review (high value > 10L) |
| APP100303 | 2,00,000 | 0.82 | human_review (high risk > 0.7) |
| APP100304 | 8,00,000 | 0.18 | auto_process |


In [ ]:
test_loans = [
    {"application_id": "APP100301", "loan_amount": 500_000,
     "applicant_summary": "Salaried, credit score 720, DTI 30%",
     "ai_recommendation": "Likely approve", "risk_score": 0.25,
     "review_required": False, "human_decision": None, "reviewer_id": None, "review_notes": None},

    {"application_id": "APP100302", "loan_amount": 1_500_000,
     "applicant_summary": "Self-employed, credit score 660, DTI 45%",
     "ai_recommendation": "Borderline", "risk_score": 0.62,
     "review_required": False, "human_decision": None, "reviewer_id": None, "review_notes": None},

    {"application_id": "APP100303", "loan_amount": 200_000,
     "applicant_summary": "Salaried, credit score 580, DTI 55%",
     "ai_recommendation": "High risk", "risk_score": 0.82,
     "review_required": False, "human_decision": None, "reviewer_id": None, "review_notes": None},

    {"application_id": "APP100304", "loan_amount": 800_000,
     "applicant_summary": "Salaried, credit score 750, DTI 25%",
     "ai_recommendation": "Likely approve", "risk_score": 0.18,
     "review_required": False, "human_decision": None, "reviewer_id": None, "review_notes": None},
]
print(f"✅ {len(test_loans)} loans loaded")


## Cell 14 — Phase 1: Run Until `interrupt()`
Graph runs `planner → executor → verifier → auto_process → human_review`.  
Execution **pauses** at `interrupt()` inside `human_review`.


In [ ]:
thread_config = {"configurable": {"thread_id": "loan_batch_001"}}

initial_state: LoanState = {
    "goal":            "Evaluate and route a batch of loan applications using risk-based HITL workflow",
    "tasks":           [], "results":  [], "critique": "",
    "approved":        False, "iterations": 0,
    "loans":           test_loans, "review_queue": [],
    "auto_queue":      [], "final_decisions": [],
}

print("🚀 Starting workflow...")
print("=" * 60)

result_phase1 = app.invoke(initial_state, config=thread_config)

print("\n" + "=" * 60)
print(f"⏸  Graph paused.")
print(f"   Auto-processed : {len(result_phase1.get('final_decisions', []))} loans")
print(f"   Pending review : {len(result_phase1.get('review_queue', []))} loans")


## Cell 15 — Phase 2: Human Decisions + Resume
Simulate underwriter decisions and resume with `Command(resume=...)`.  
Rule: `risk_score < 0.70` → APPROVE, else → REJECT.


In [ ]:
review_queue = result_phase1.get("review_queue", [])

simulated_decisions = [
    {
        "application_id": loan["application_id"],
        "decision":  ReviewDecision.APPROVE.value if loan["risk_score"] < HIGH_RISK_SCORE else ReviewDecision.REJECT.value,
        "reviewer_id": f"UNDERWRITER_{i+1:03d}",
        "notes":     "Manual review complete",
    }
    for i, loan in enumerate(review_queue)
]

print("👤 Simulated decisions:")
for d in simulated_decisions:
    icon = "✅" if d["decision"] == "approve" else "❌"
    print(f"   {icon} [{d['application_id']}] → {d['decision'].upper()} by {d['reviewer_id']}")

print("\n▶  Resuming graph...")
result_final = app.invoke(Command(resume={"decisions": simulated_decisions}), config=thread_config)
print("✅ Graph completed.")


## Cell 16 — Final Summary

In [ ]:
final_decisions = result_final.get("final_decisions", [])
icons = {"approve": "✅", "reject": "❌", "escalate": "⚠️"}

print("\n" + "═" * 70)
print("  FINAL DECISIONS")
print("═" * 70)
print(f"{'ID':<12} {'Amount (INR)':>14} {'Risk':>6}  {'Route':<14} {'Decision'}")
print("─" * 70)
for d in sorted(final_decisions, key=lambda x: x["application_id"]):
    dec   = d.get("human_decision", "unknown")
    route = "human_review" if d.get("review_required") else "auto_process"
    print(f"{d['application_id']:<12} {d['loan_amount']:>14,.0f} {d['risk_score']:>6.2f}  {route:<14} {icons.get(dec,'❓')} {dec}")
print("─" * 70)
counts = {}
for d in final_decisions:
    k = d.get("human_decision","unknown").upper()
    counts[k] = counts.get(k,0) + 1
print("  " + "  |  ".join(f"{k}: {v}" for k,v in counts.items()))
print("═" * 70)
